In [99]:
from google.colab import drive
import pickle
import pandas as pd
from collections import defaultdict
import random
from torchvision import transforms
from torch.utils.data import Dataset
from PIL import Image
import os
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader
import timm
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [100]:
!pip install rouge_score

In [101]:
!pip install pycocoevalcap

In [102]:
import nltk
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [103]:
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer

from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.spice.spice import Spice

import pandas as pd
import torch

In [104]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [105]:
# Copying 8k to Colabs local SSD
if not os.path.exists("/content/flickr8k.zip"):
  !cp -r "/content/drive/MyDrive/MMRetrieval/flickr8k.zip" /content/

In [106]:
if not os.path.isdir("/content/flickr8k/Images"):
  !unzip "/content/flickr8k.zip" -d "/content/flickr8k"

In [107]:
print("Images:", len(os.listdir("/content/flickr8k/Images")))

Images: 8091


In [108]:
# Copying 30k to Colabs local SSD
if not os.path.exists("/content/flickr30k.zip"):
  !cp -r "/content/drive/MyDrive/MMRetrieval/flickr30k.zip" /content/

In [109]:
if not os.path.isdir("/content/flickr30k/Images"):
  !unzip "/content/flickr30k.zip" -d "/content/flickr30k"

In [110]:
print("Images:", len(os.listdir("/content/flickr30k/Images")))

Images: 31811


In [111]:
DATASETS = {
    "flickr8k": {
        "ROOT": "/content/flickr8k",
        "IMAGE_DIR": "/content/flickr8k/Images",
        "CAPTION_FILE": "/content/flickr8k/captions.txt",
        "flickr_split": "/content/drive/MyDrive/MMRetrieval/Preprocessing/flickr8k/flickr8k_split.pkl",
        "vocab":"/content/drive/MyDrive/MMRetrieval/Preprocessing/flickr8k/vocab.pkl"
    },
    "flickr30k": {
        "ROOT": "/content/flickr30k",
        "IMAGE_DIR": "/content/flickr30k/Images",
        "CAPTION_FILE": "/content/flickr30k/captions.txt",
        "flickr_split": "/content/drive/MyDrive/MMRetrieval/Preprocessing/flickr30k/flickr30k_split.pkl",
        "vocab":"/content/drive/MyDrive/MMRetrieval/Preprocessing/flickr30k/vocab.pkl"
    }
}

In [112]:
def image_caption_map(train_df,val_df,test_df):
  train_caption_map = defaultdict(list)
  val_caption_map = defaultdict(list)
  test_caption_map = defaultdict(list)

  bad_img = "861608773_bdafd5c996.jpg"

  train_caption_map.pop(bad_img, None)
  val_caption_map.pop(bad_img, None)
  test_caption_map.pop(bad_img, None)

  for _, row in train_df.iterrows():
      train_caption_map[row["image"]].append(row["caption"])

  for _, row in val_df.iterrows():
      val_caption_map[row["image"]].append(row["caption"])

  for _, row in test_df.iterrows():
      test_caption_map[row["image"]].append(row["caption"])
  return train_caption_map,val_caption_map,test_caption_map



In [113]:
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(
        224,
        scale=(0.8, 1.0),
        ratio=(0.9, 1.1)
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(5),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.05
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

In [114]:
image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [115]:
# Encoding

def encode_caption(text, vocab):

    tokens = text.split()

    encoded = [vocab["<SOS>"]]

    for token in tokens:
        encoded.append(
            vocab.get(token, vocab["<UNK>"])
        )

    encoded.append(vocab["<EOS>"])

    return encoded

In [116]:
class FlickrCaptionDataset(Dataset):

    def __init__(
            self,
            caption_map,
            image_dir,
            vocab,
            transform=None,
            random_caption=True):

        self.image_names = sorted(caption_map.keys())
        self.caption_map = caption_map
        self.image_dir = image_dir
        self.vocab = vocab
        self.transform = transform
        self.random_caption = random_caption

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):

        image_name = self.image_names[idx]

        captions = self.caption_map[image_name]

        if self.random_caption:
            caption = random.choice(captions)
        else:
            caption = captions[0]

        try:
            image = Image.open(
                os.path.join(self.image_dir, image_name)
            ).convert("RGB")
        except Exception:
            return self.__getitem__((idx + 1) % len(self))

        if self.transform:
            image = self.transform(image)

        caption = encode_caption(caption, self.vocab)

        input_caption = torch.tensor(
            caption[:-1],
            dtype=torch.long
        )

        target_caption = torch.tensor(
            caption[1:],
            dtype=torch.long
        )

        return image, input_caption, target_caption

In [117]:
from torch.utils.data import Dataset
from PIL import Image
import os

class FlickrCaptionTestDataset(Dataset):

    def __init__(
            self,
            caption_map,
            image_dir,
            transform=None
    ):

        self.image_names = sorted(caption_map.keys())
        self.caption_map = caption_map
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):

        image_name = self.image_names[idx]

        # List containing all 5 captions
        captions = self.caption_map[image_name]

        try:
            image = Image.open(
                os.path.join(self.image_dir, image_name)
            ).convert("RGB")

        except Exception:
            return self.__getitem__(
                (idx + 1) % len(self)
            )

        if self.transform:
            image = self.transform(image)

        return image, captions

In [118]:
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):

    images = []
    input_captions = []
    target_captions_list = []

    for image, input_caption, target_caption in batch:
        images.append(image)
        input_captions.append(input_caption)
        target_captions_list.append(target_caption)

    images = torch.stack(images)

    input_captions = pad_sequence(
        input_captions,
        batch_first=True,
        padding_value=vocab["<PAD>"]
    )

    padded_target_captions = pad_sequence(
        target_captions_list,
        batch_first=True,
        padding_value=vocab["<PAD>"]
    )

    return images, input_captions, padded_target_captions

In [119]:
# Encoder Image
class ResnetImageEncoder(nn.Module):

    def __init__(self, embed_dim=512):
        super().__init__()

        backbone = models.resnet50(
            weights=models.ResNet50_Weights.IMAGENET1K_V2
        )

        # Freeze initially
        for param in backbone.parameters():
            param.requires_grad = False

        # Fine-tune only deeper layers
        for param in backbone.layer4.parameters():
            param.requires_grad = True

        self.backbone = nn.Sequential(
            *list(backbone.children())[:-2]
        )

        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        self.projection = nn.Sequential(
            nn.Linear(2048, 1024),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(1024, embed_dim),
            nn.LayerNorm(embed_dim)
        )

        for m in self.projection:
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, images):

        features = self.backbone(images)

        # (B,2048,7,7)

        features = self.pool(features)

        # (B,2048,1,1)

        features = torch.flatten(features, 1)

        # (B,2048)

        features = self.projection(features)

        # (B,512)

        return features

In [120]:
#decoder
class LSTMCaptionDecoder(nn.Module):

    def __init__(
        self,
        vocab_size,
        embed_dim=512,
        hidden_dim=512,
        pad_idx=0
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=pad_idx
        )

        self.embedding_dropout = nn.Dropout(0.3)

        # Initialize LSTM from image feature
        self.init_h = nn.Linear(embed_dim, hidden_dim)
        self.init_c = nn.Linear(embed_dim, hidden_dim)

        self.h_norm = nn.LayerNorm(hidden_dim)
        self.c_norm = nn.LayerNorm(hidden_dim)

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True
        )

        self.output_dropout = nn.Dropout(0.3)

        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, vocab_size)
        )

        # -----------------------
        # Initialization
        # -----------------------

        nn.init.xavier_uniform_(self.embedding.weight)

        with torch.no_grad():
            self.embedding.weight[pad_idx].fill_(0)

        for m in self.fc:
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

        nn.init.xavier_uniform_(self.init_h.weight)
        nn.init.zeros_(self.init_h.bias)

        nn.init.xavier_uniform_(self.init_c.weight)
        nn.init.zeros_(self.init_c.bias)

        for name, param in self.lstm.named_parameters():

            if "weight_ih" in name:
                nn.init.xavier_uniform_(param)

            elif "weight_hh" in name:
                nn.init.orthogonal_(param)

            elif "bias" in name:
                nn.init.zeros_(param)

                # Forget gate bias = 1
                hidden = param.size(0) // 4
                param.data[hidden:2*hidden].fill_(1)

    def forward(
        self,
        image_features,
        captions
    ):

        embeddings = self.embedding(captions)

        embeddings = self.embedding_dropout(embeddings)

        h0 = torch.tanh(
            self.h_norm(
                self.init_h(image_features)
            )
        ).unsqueeze(0)

        c0 = torch.tanh(
            self.c_norm(
                self.init_c(image_features)
            )
        ).unsqueeze(0)

        outputs, _ = self.lstm(
            embeddings,
            (h0, c0)
        )

        outputs = self.output_dropout(outputs)

        outputs = self.fc(outputs)

        return outputs

In [121]:
#joint model
class ResnetLSTMImageCaptioningModel(nn.Module):

    def __init__(self, encoder, decoder):
        super().__init__()

        self.encoder = encoder
        self.decoder = decoder

    def forward(self, images, captions):

        image_features = self.encoder(images)

        outputs = self.decoder(
            image_features,
            captions
        )

        return outputs

    @staticmethod
    def length_penalty(length, alpha=0.7):
        return ((5 + length) ** alpha) / ((5 + 1) ** alpha)

    def generate_caption(
        self,
        images,
        vocab,
        max_length=30,
        beam_size=5
    ):

        self.eval()

        batch_size = images.size(0)

        assert batch_size == 1, \
            "Beam search currently supports batch_size=1."

        with torch.no_grad():

            image_features = self.encoder(images)

            h = torch.tanh(
                self.decoder.h_norm(
                    self.decoder.init_h(image_features)
                )
            ).unsqueeze(0)

            c = torch.tanh(
                self.decoder.c_norm(
                    self.decoder.init_c(image_features)
                )
            ).unsqueeze(0)

            beams = [
                (
                    [vocab["<SOS>"]],
                    0.0,
                    (h, c)
                )
            ]

            completed = []

            for step in range(max_length):

                candidates = []

                for seq, score, hidden in beams:

                    last_token = seq[-1]

                    if last_token == vocab["<EOS>"]:
                        completed.append((seq, score))
                        continue

                    current_word = torch.tensor(
                        [last_token],
                        device=images.device
                    )

                    embedding = self.decoder.embedding(
                        current_word
                    )

                    embedding = self.decoder.embedding_dropout(
                        embedding
                    )

                    output, hidden_new = self.decoder.lstm(
                        embedding.unsqueeze(1),
                        hidden
                    )

                    scores = self.decoder.fc(
                        self.decoder.output_dropout(
                            output.squeeze(1)
                        )
                    )

                    # Block unwanted tokens
                    scores[:, vocab["<PAD>"]] = -1e9
                    scores[:, vocab["<SOS>"]] = -1e9

                    if "<UNK>" in vocab:
                        scores[:, vocab["<UNK>"]] = -1e9

                    # Prevent early EOS
                    if step < 3:
                        scores[:, vocab["<EOS>"]] = -1e9

                    log_probs = F.log_softmax(
                        scores,
                        dim=1
                    )

                    top_log_probs, top_indices = torch.topk(
                        log_probs,
                        beam_size,
                        dim=1
                    )

                    for k in range(beam_size):

                        token = top_indices[0, k].item()

                        new_seq = seq + [token]

                        new_score = score + \
                            top_log_probs[0, k].item()

                        # Clone hidden state
                        h_new = hidden_new[0].clone()
                        c_new = hidden_new[1].clone()

                        candidates.append(
                            (
                                new_seq,
                                new_score,
                                (h_new, c_new)
                            )
                        )

                if len(candidates) == 0:
                    break

                candidates = sorted(
                    candidates,
                    key=lambda x:
                        x[1] /
                        self.length_penalty(len(x[0])),
                    reverse=True
                )

                beams = candidates[:beam_size]

            if len(completed) == 0:
                completed = [
                    (
                        beams[0][0],
                        beams[0][1]
                    )
                ]

            completed = sorted(
                completed,
                key=lambda x:
                    x[1] /
                    self.length_penalty(len(x[0])),
                reverse=True
            )

            best_seq = completed[0][0]

            return [best_seq]

In [122]:
def build_image_text_decoder(vocab,device):
  encoder = ResnetImageEncoder(embed_dim=512)

  decoder = LSTMCaptionDecoder(
      vocab_size=len(vocab),
      embed_dim=512,
      hidden_dim=512,
      pad_idx=vocab["<PAD>"]
  )

  model = ResnetLSTMImageCaptioningModel(
      encoder,
      decoder
  ).to(device)

  return model

In [123]:
def define_optimizer(model):

    optimizer = torch.optim.AdamW(

        [

            # Fine-tuned ResNet
            {
                "params": filter(
                    lambda p: p.requires_grad,
                    model.encoder.backbone.parameters()
                ),
                "lr": 1e-5
            },

            # Projection layer
            {
                "params": model.encoder.projection.parameters(),
                "lr": 2e-4
            },

            # Embedding
            {
                "params": model.decoder.embedding.parameters(),
                "lr": 2e-4
            },

            # LSTM
            {
                "params": model.decoder.lstm.parameters(),
                "lr": 2e-4
            },

            # Hidden state initialization
            {
                "params": (
                    list(model.decoder.init_h.parameters()) +
                    list(model.decoder.init_c.parameters())
                ),
                "lr": 2e-4
            },

            # Vocabulary prediction head
            {
                "params": model.decoder.fc.parameters(),
                "lr": 3e-4
            }

        ],

        weight_decay=1e-4

    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=50
    )

    return optimizer, scheduler

In [124]:
def evaluate(
    model,
    dataloader,
    criterion,
    device
):

    model.eval()

    total_loss = 0.0
    total_tokens = 0

    with torch.inference_mode():

        for images, input_captions, target_captions in dataloader:

            images = images.to(device, non_blocking=True)
            input_captions = input_captions.to(device, non_blocking=True)
            target_captions = target_captions.to(device, non_blocking=True)

            outputs = model(
                images,
                input_captions
            )

            loss = criterion(
                outputs.reshape(-1, outputs.size(-1)),
                target_captions.reshape(-1)
            )

            # Count only non-PAD tokens
            valid_tokens = (
                target_captions != criterion.ignore_index
            ).sum().item()

            total_loss += loss.item() * valid_tokens
            total_tokens += valid_tokens

    return total_loss / total_tokens

In [125]:
import os
import time
import torch
import torch.nn as nn

def model_training(
        model,
        train_loader,
        val_loader,
        optimizer,
        scheduler,
        criterion,
        dataset_name):

    NUM_EPOCHS = 50

    best_val_loss = float("inf")

    patience = 8
    epochs_without_improvement = 0

    SAVE_DIR = f"/content/drive/MyDrive/MMCaptioning/C1/{dataset_name}"
    os.makedirs(SAVE_DIR, exist_ok=True)

    BEST_MODEL_PATH = os.path.join(
        SAVE_DIR,
        "best_captioning_C1_model.pth"
    )

    for epoch in range(NUM_EPOCHS):

        print(f"\n================ Epoch {epoch+1}/{NUM_EPOCHS} ================")

        model.train()

        running_loss = 0.0

        epoch_start = time.time()

        for batch_idx, (images, input_captions, target_captions) in enumerate(train_loader):

            batch_start = time.time()

            images = images.to(device, non_blocking=True)
            input_captions = input_captions.to(device, non_blocking=True)
            target_captions = target_captions.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            outputs = model(
                images,
                input_captions
            )

            loss = criterion(
                outputs.reshape(-1, outputs.size(-1)),
                target_captions.reshape(-1)
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            optimizer.step()

            running_loss += loss.item()

            print(
                f"Batch {batch_idx+1:03d}/{len(train_loader)} | "
                f"Loss: {loss.item():.4f} | "
                f"Time: {time.time()-batch_start:.2f}s"
            )

        train_loss = running_loss / len(train_loader)

        train_time = time.time() - epoch_start

        # Validation

        val_start = time.time()

        val_loss = evaluate(
            model,
            val_loader,
            criterion,
            device
        )

        scheduler.step()

        val_time = time.time() - val_start

        print("\n---------------- Summary ----------------")

        print(f"Train Loss      : {train_loss:.4f}")
        print(f"Validation Loss : {val_loss:.4f}")
        print(f"Training Time   : {train_time:.2f}s")
        print(f"Validation Time : {val_time:.2f}s")

        print(
            f"GPU Memory Used : "
            f"{torch.cuda.memory_allocated()/1024**3:.2f} GB"
        )

        # Save Best

        if val_loss < best_val_loss:

            best_val_loss = val_loss
            epochs_without_improvement = 0

            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict(),
                    "epoch": epoch,
                    "val_loss": val_loss
                },
                BEST_MODEL_PATH
            )

            print(f"✓ Best model saved (Val Loss: {val_loss:.4f})")

        else:

            epochs_without_improvement += 1

            print(
                f"No improvement for "
                f"{epochs_without_improvement}/{patience} epochs"
            )

        if epochs_without_improvement >= patience:

            print("\nEarly stopping triggered!")

            break

    checkpoint = torch.load(
        BEST_MODEL_PATH,
        map_location=device
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    return model, BEST_MODEL_PATH

In [126]:
import gc
def reset_resources():
  gc.collect()
  torch.cuda.empty_cache()
  print("\nMemory after cleanup")
  print("Allocated:",torch.cuda.memory_allocated()/1024**3)
  print("Reserved:",torch.cuda.memory_reserved()/1024**3)

In [127]:
def caption_test_collate_fn(batch):

    images = []
    all_references = []

    for image, references in batch:
        images.append(image)
        all_references.append(references)

    images = torch.stack(images)

    return images, all_references

In [128]:
def prepare_caption_dataloaders(
        train_caption_map,
        val_caption_map,
        test_caption_map):

    BATCH_SIZE = 192

    train_dataset = FlickrCaptionDataset(
        train_caption_map,
        IMAGE_DIR,
        vocab,
        train_transform,
        random_caption=True
    )

    val_dataset = FlickrCaptionDataset(
        val_caption_map,
        IMAGE_DIR,
        vocab,
        image_transform,
        random_caption=False
    )

    # Different dataset for evaluation
    test_dataset = FlickrCaptionTestDataset(
        test_caption_map,
        IMAGE_DIR,
        image_transform
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=8,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=8,
        pin_memory=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=caption_test_collate_fn,   # Different collate function
        num_workers=8,
        pin_memory=True
    )

    return train_loader, val_loader, test_loader

In [129]:
def model_testing(model,checkpoint_path,test_loader,vocab,device):
  checkpoint = torch.load(checkpoint_path,map_location=device)
  model.load_state_dict(checkpoint["model_state_dict"])
  model.to(device)
  model.eval()



  idx2word = {v: k for k, v in vocab.items()}

  references = []
  predictions = []

  rouge = rouge_scorer.RougeScorer(["rougeL"],use_stemmer=True)

  rouge_scores = []
  meteor_scores = []

  coco_refs = {}
  coco_preds = {}

  sample_id = 0

  print("Generating captions...")

  with torch.inference_mode():

      for images, all_references in test_loader:

        images = images.to(device)   # move BEFORE running encoder

        predicted_ids = []

        for j in range(images.size(0)):

            pred = model.generate_caption(
                images[j:j+1],
                vocab,
                max_length=30,
                beam_size=5
            )

            predicted_ids.extend(pred)

        for i in range(images.size(0)):
            # Prediction
            pred_words = []
            for idx in predicted_ids[i]:
                word = idx2word[idx]
                if word == "<EOS>":
                    break
                if word not in ["<SOS>", "<PAD>"]:
                    pred_words.append(word)

            pred_sentence = " ".join(pred_words)

            # References (5 captions)
            ref_words = [ref.split() for ref in all_references[i]]
            references.append(ref_words)
            predictions.append(pred_words)

            # ROUGE-L
            rouge_values = []

            for ref in all_references[i]:

                rouge_values.append(rouge.score(ref,pred_sentence)["rougeL"].fmeasure)

            rouge_scores.append(max(rouge_values))

            # METEOR
            meteor_scores.append(meteor_score(ref_words,pred_words))

            # COCO format
            coco_refs[sample_id] = [str(ref) for ref in all_references[i]]

            coco_preds[sample_id] = [str(pred_sentence)]

            sample_id += 1

  # BLEU
  smoothie = SmoothingFunction().method4
  bleu1 = corpus_bleu(references,predictions,weights=(1,0,0,0),smoothing_function=smoothie)

  bleu2 = corpus_bleu(references,predictions,weights=(0.5,0.5,0,0),smoothing_function=smoothie)

  bleu3 = corpus_bleu(references,predictions,weights=(0.33,0.33,0.33,0),smoothing_function=smoothie)

  bleu4 = corpus_bleu(references,predictions,weights=(0.25,0.25,0.25,0.25),smoothing_function=smoothie)

  # CIDEr
  cider_scorer = Cider()

  cider_score, _ = cider_scorer.compute_score(coco_refs,coco_preds)

  # SPICE

  # Results

  results = pd.DataFrame({

      "Metric":[
          "BLEU-1",
          "BLEU-2",
          "BLEU-3",
          "BLEU-4",
          "ROUGE-L",
          "METEOR",
          "CIDEr",

      ],

      "Score":[
          bleu1,
          bleu2,
          bleu3,
          bleu4,
          sum(rouge_scores)/len(rouge_scores),
          sum(meteor_scores)/len(meteor_scores),
          cider_score,

      ]

  })
  print(coco_refs[0])
  print(coco_preds[0])
  display(results)

  return results

In [130]:
for dataset_name, cfg in DATASETS.items():

    torch.cuda.reset_peak_memory_stats()

    print("\n************************")
    print(f"\nProcessing - {dataset_name}")
    print("\n************************\n")

    ROOT = cfg["ROOT"]
    IMAGE_DIR = cfg["IMAGE_DIR"]
    CAPTION_FILE = cfg["CAPTION_FILE"]

    with open(cfg["flickr_split"], "rb") as f:
        split = pickle.load(f)

    with open(cfg["vocab"], "rb") as f:
        vocab = pickle.load(f)


    #Build Model

    model = build_image_text_decoder(
        vocab,
        device
    )

    optimizer, scheduler = define_optimizer(model)

    # Dataset

    df = pd.read_csv(CAPTION_FILE)

    train_imgs = split["train"]
    val_imgs = split["val"]
    test_imgs = split["test"]

    print(
        f"Train: {len(train_imgs)} | "
        f"Val: {len(val_imgs)} | "
        f"Test: {len(test_imgs)}"
    )

    train_df = df[df["image"].isin(train_imgs)].reset_index(drop=True)
    val_df = df[df["image"].isin(val_imgs)].reset_index(drop=True)
    test_df = df[df["image"].isin(test_imgs)].reset_index(drop=True)

    train_df = train_df.dropna(subset=["caption"]).reset_index(drop=True)
    val_df = val_df.dropna(subset=["caption"]).reset_index(drop=True)
    test_df = test_df.dropna(subset=["caption"]).reset_index(drop=True)

    train_caption_map, val_caption_map, test_caption_map = image_caption_map(
        train_df,
        val_df,
        test_df
    )

    train_loader, val_loader, test_loader = prepare_caption_dataloaders(
        train_caption_map,
        val_caption_map,
        test_caption_map
    )
    criterion = nn.CrossEntropyLoss(ignore_index=vocab["<PAD>"],label_smoothing=0.1)

    #Train

    best_model, FINAL_MODEL_PATH = model_training(
        model,
        train_loader,
        val_loader,
        optimizer,
        scheduler,
        criterion,
        dataset_name
    )

    #Test

    model_testing(
        best_model,
        FINAL_MODEL_PATH,
        test_loader,
        vocab,
        device
    )

    # Cleanup

    del model
    del optimizer
    del scheduler

    del train_loader
    del val_loader
    del test_loader

    del train_df
    del val_df
    del test_df

    del train_caption_map
    del val_caption_map
    del test_caption_map

    del split
    del vocab
    del df

    reset_resources()



************************

Processing - flickr8k

************************

Train: 6068 | Val: 1011 | Test: 1012

================ Epoch 1/50 ================
Batch 001/32 | Loss: 7.8554 | Time: 0.11s
Batch 002/32 | Loss: 7.8012 | Time: 0.10s
Batch 003/32 | Loss: 7.7439 | Time: 0.09s
Batch 004/32 | Loss: 7.6692 | Time: 0.09s
Batch 005/32 | Loss: 7.5748 | Time: 0.09s
Batch 006/32 | Loss: 7.4649 | Time: 0.09s
Batch 007/32 | Loss: 7.3169 | Time: 0.09s
Batch 008/32 | Loss: 7.1494 | Time: 0.09s
Batch 009/32 | Loss: 6.9428 | Time: 0.10s
Batch 010/32 | Loss: 6.7698 | Time: 0.10s
Batch 011/32 | Loss: 6.5523 | Time: 0.10s
Batch 012/32 | Loss: 6.3521 | Time: 0.10s
Batch 013/32 | Loss: 6.1533 | Time: 0.09s
Batch 014/32 | Loss: 6.0123 | Time: 0.09s
Batch 015/32 | Loss: 5.8523 | Time: 0.09s
Batch 016/32 | Loss: 5.8716 | Time: 0.09s
Batch 017/32 | Loss: 5.7646 | Time: 0.10s
Batch 018/32 | Loss: 5.7113 | Time: 0.10s
Batch 019/32 | Loss: 5.5918 | Time: 0.10s
Batch 020/32 | Loss: 5.5406 | Time: 0.09s
B

,Metric,Score
0,BLEU-1,0.535045
1,BLEU-2,0.358416
2,BLEU-3,0.236890
3,BLEU-4,0.149452
4,ROUGE-L,0.449882
5,METEOR,0.378598
6,CIDEr,0.412919


Streaming output truncated to the last 5000 lines.
GPU Memory Used : 0.83 GB
✓ Best model saved (Val Loss: 4.1282)

================ Epoch 14/50 ================
Batch 001/125 | Loss: 4.0443 | Time: 0.11s
Batch 002/125 | Loss: 4.0759 | Time: 0.11s
Batch 003/125 | Loss: 4.0315 | Time: 0.12s
Batch 004/125 | Loss: 3.9811 | Time: 0.10s
Batch 005/125 | Loss: 3.9691 | Time: 0.10s
Batch 006/125 | Loss: 4.0326 | Time: 0.10s
Batch 007/125 | Loss: 4.0250 | Time: 0.10s
Batch 008/125 | Loss: 3.9425 | Time: 0.11s
Batch 009/125 | Loss: 4.0215 | Time: 0.10s
Batch 010/125 | Loss: 3.9989 | Time: 0.10s
Batch 011/125 | Loss: 3.9873 | Time: 0.10s
Batch 012/125 | Loss: 3.9214 | Time: 0.10s
Batch 013/125 | Loss: 4.0239 | Time: 0.10s
Batch 014/125 | Loss: 4.0264 | Time: 0.10s
Batch 015/125 | Loss: 4.0248 | Time: 0.10s
Batch 016/125 | Loss: 3.9576 | Time: 0.10s
Batch 017/125 | Loss: 3.9945 | Time: 0.11s
Batch 018/125 | Loss: 3.9824 | Time: 0.11s
Batch 019/125 | Loss: 3.9286 | Time: 0.10s
Batch 020/125 | Loss:

,Metric,Score
0,BLEU-1,0.466214
1,BLEU-2,0.311108
2,BLEU-3,0.205217
3,BLEU-4,0.128546
4,ROUGE-L,0.398789
5,METEOR,0.347604
6,CIDEr,0.283623



Memory after cleanup
Allocated: 0.37775564193725586
Reserved: 1.99609375
